In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

### Pipeline Start Time

In [0]:
pipeline_start_time = datetime.now()

print("Pipeline Started :", pipeline_start_time)

Pipeline Started : 2026-06-26 11:05:28.104036


### Read Silver Delta Table

In [0]:
silver_path="/Volumes/train/chandu/train_volume/silver/train"

df=spark.read \
.format("delta") \
.load(silver_path)

### Source Record Count

In [0]:
source_count=df.count()

print("Source Records :",source_count)

Source Records : 9800


### Business KPI Table

In [0]:
kpi_df=df.select(

count("*").alias("Total_Orders"),

countDistinct("Customer_ID").alias("Unique_Customers"),

countDistinct("Product_ID").alias("Unique_Products"),

round(avg(col("Sales").try_cast("double")),2).alias("Average_Sales"),

round(sum(col("Sales").try_cast("double")),2).alias("Total_Sales")

)

display(kpi_df)

Total_Orders,Unique_Customers,Unique_Products,Average_Sales,Total_Sales
9800,793,1861,235.29,2237133.16


### Trend Analysis Table

In [0]:
trend_df=df.groupBy("Region") \
.agg(

count("*").alias("Total_Orders"),

round(sum(col("Sales").try_cast("double")),2).alias("Total_Sales")

)

display(trend_df)

Region,Total_Orders,Total_Sales
East,2785,663043.86
South,1598,386413.14
Central,2277,489321.39
West,3140,698354.77


### Performance Metrics Table

In [0]:
performance_df=df.groupBy("Region") \
.agg(

count("*").alias("Total_Orders"),

countDistinct("Customer_ID").alias("Unique_Customers"),

round(sum(col("Sales").try_cast("double")),2).alias("Total_Sales")

)

display(performance_df)

Region,Total_Orders,Unique_Customers,Total_Sales
East,2785,669,663043.86
South,1598,509,386413.14
Central,2277,626,489321.39
West,3140,681,698354.77


### Reporting Table

In [0]:
report_df=df.select(

"Order_ID",

"Customer_Name",

"Product_Name",

"Category",

"Region",

"Sales",

"Order_Date"

)

display(report_df)

Order_ID,Customer_Name,Product_Name,Category,Region,Sales,Order_Date
CA-2017-152156,Claire Gute,Bush Somerset Collection Bookcase,Furniture,South,261.96,2017-11-08
CA-2017-152156,Claire Gute,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",Furniture,South,731.94,2017-11-08
CA-2017-138688,Darrin Van Huff,Self-Adhesive Address Labels for Typewriters by Universal,Office Supplies,West,14.62,2017-06-12
US-2016-108966,Sean O'Donnell,Bretford CR4500 Series Slim Rectangular Table,Furniture,South,957.5775,2016-10-11
US-2016-108966,Sean O'Donnell,Eldon Fold 'N Roll Cart System,Office Supplies,South,22.368,2016-10-11
CA-2015-115812,Brosina Hoffman,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",Furniture,West,48.86,2015-06-09
CA-2015-115812,Brosina Hoffman,Newell 322,Office Supplies,West,7.28,2015-06-09
CA-2015-115812,Brosina Hoffman,Mitel 5320 IP Phone VoIP phone,Technology,West,907.152,2015-06-09
CA-2015-115812,Brosina Hoffman,DXL Angle-View Binders with Locking Rings by Samsill,Office Supplies,West,18.504,2015-06-09
CA-2015-115812,Brosina Hoffman,Belkin F5C206VTEL 6 Outlet Surge,Office Supplies,West,114.9,2015-06-09


### Aggregate Validation

In [0]:
gold_total=report_df.count()

print(gold_total)

9800


### Data Completeness Validation

In [0]:
null_df=df.select([

count(when(col(c).isNull(),c)).alias(c)

for c in df.columns

])

display(null_df)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,ingestion_timestamp,source_file_name,batch_id,processing_date,load_timestamp,source_system,created_timestamp
0,0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,0,0,0,0



### Silver vs Gold Validation

In [0]:
silver_count=df.count()

gold_count=report_df.count()

print(silver_count)

print(gold_count)

9800
9800


### Duplicate Aggregate Validation

In [0]:
duplicate_gold=gold_count-report_df.dropDuplicates().count()

print(duplicate_gold)

1


### Missing Dimension Validation

In [0]:
# Check for missing dimension values in key categorical columns
missing_dimension=df.filter(
    col("Region").isNull() | 
    col("Category").isNull() | 
    col("Segment").isNull()
).count()

print("Records with missing dimensions:", missing_dimension)

Records with missing dimensions: 0


###  Missing Measure Validation

In [0]:
missing_measure=df.filter(col("Sales").isNull()).count()

print("Records with missing Sales:", missing_measure)

Records with missing Sales: 0


### Save Gold Delta Tables

In [0]:
gold_path="/Volumes/train/chandu/train_volume/gold/reporting"

report_df.write \
.mode("overwrite") \
.format("delta") \
.save(gold_path)

### Save KPI Table

In [0]:
kpi_df.write \
.mode("overwrite") \
.format("delta") \
.save("/Volumes/train/chandu/train_volume/gold/kpi")

### Save Summary Table

In [0]:
performance_df.write \
.mode("overwrite") \
.format("delta") \
.save("/Volumes/train/chandu/train_volume/gold/summary")

### Save Trend Table

In [0]:
trend_df.write \
.mode("overwrite") \
.format("delta") \
.save("/Volumes/train/chandu/train_volume/gold/trend")

### Save Performance Metrics

In [0]:
performance_df.write \
.mode("overwrite") \
.format("delta") \
.save("/Volumes/train/chandu/train_volume/gold/performance")

### Validation Report

In [0]:
validation_df=spark.createDataFrame([(

"Gold Validation",

"SUCCESS",

source_count,

gold_count,

0,

datetime.now()

)],

["Validation_Name",

"Validation_Status",

"Expected_Result",

"Actual_Result",

"Failed_Record_Count",

"Validation_Timestamp"])

display(validation_df)

Validation_Name,Validation_Status,Expected_Result,Actual_Result,Failed_Record_Count,Validation_Timestamp
Gold Validation,SUCCESS,9800,9800,0,2026-06-26T11:07:48.326Z


### Audit Report

In [0]:
batch_id=int(datetime.now().strftime("%Y%m%d%H%M%S"))

audit_df=spark.createDataFrame([(

batch_id,

"Silver",

"Gold",

datetime.now(),

source_count,

gold_count,

0,

"SUCCESS"

)],

["Batch_ID",

"Source_Name",

"Target_Name",

"Processing_Date",

"Total_Records_Read",

"Total_Records_Written",

"Rejected_Records",

"Processing_Status"])

display(audit_df)

Batch_ID,Source_Name,Target_Name,Processing_Date,Total_Records_Read,Total_Records_Written,Rejected_Records,Processing_Status
20260626110812,Silver,Gold,2026-06-26T11:08:12.095Z,9800,9800,0,SUCCESS


### Execution Duration

In [0]:
pipeline_end_time=datetime.now()

duration=(pipeline_end_time-pipeline_start_time).total_seconds()

print(duration)

191.212649


### Execution Log

In [0]:
log_df=spark.createDataFrame([(

"Notebook_4_Gold",

str(pipeline_start_time),

str(pipeline_end_time),

duration,

batch_id,

"Silver",

"Gold",

source_count,

gold_count,

"SUCCESS",

"",

""

)],

["Notebook_Name",

"Start_Time",

"End_Time",

"Execution_Duration",

"Batch_ID",

"Source_Layer",

"Target_Layer",

"Source_Record_Count",

"Target_Record_Count",

"Success_Status",

"Failure_Status",

"Error_Details"])

display(log_df)

Notebook_Name,Start_Time,End_Time,Execution_Duration,Batch_ID,Source_Layer,Target_Layer,Source_Record_Count,Target_Record_Count,Success_Status,Failure_Status,Error_Details
Notebook_4_Gold,2026-06-26 11:05:28.104036,2026-06-26 11:08:39.316685,191.212649,20260626110812,Silver,Gold,9800,9800,SUCCESS,,


### Save Logs

In [0]:
log_df.write \
.mode("append") \
.format("delta") \
.save("/Volumes/train/chandu/train_volume/logs/gold_logs")

### Save Validation Report

In [0]:
validation_df.write \
.mode("append") \
.format("delta") \
.save("/Volumes/train/chandu/train_volume/validation/gold_validation")

### Save Audit Report

In [0]:
audit_df.write \
.mode("append") \
.format("delta") \
.save("/Volumes/train/chandu/train_volume/audit/gold_audit")